# 3D UMAP & HDBSCAN clustering study notebook

This notebook accompanies the UROFND clustering study. It loads a csv, standardizes selected features, estimates a three-dimensional UMAP embedding, applies HDBSCAN, creates a cluster summary and figure, and exports the results. For reuse, change only the values in the **Configuration** cell unless the analytical method itself is being changed.

## Reuse

For information on reuse policies please visit: https://github.com/UROFND-repo

### Citation

    Monteiro, S., Maillard, A., Louis, E., Hentzen, C., Al Chare, I., Baltasis, S., Teng, M., Adrien, V., & Garcin, B. (2026). *Towards a Multidimensional Exploration of Functional Neurological Disorder*. Manuscript submitted to *Neurology*.

    Monteiro, S. (2026). *URO–FND Clustering Study Repo* [Research repository]. Use is governed by the repository licence.

## 1. Install dependencies

Run once in a fresh environment. Restart the kernel if requested.

In [ ]:
%pip install -q numpy pandas matplotlib pillow scikit-learn umap-learn hdbscan

## 2. Import dependencies and report versions

In [ ]:
from importlib.metadata import version
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hdbscan import HDBSCAN
from sklearn.preprocessing import StandardScaler
from umap import UMAP

PACKAGE_NAMES = [
    "numpy", "pandas", "matplotlib", "Pillow",
    "scikit-learn", "umap-learn", "hdbscan",
]

print("Package versions used in this run:")
for package_name in PACKAGE_NAMES:
    print(f"  {package_name}: {version(package_name)}")

## 3. Configuration

Replace the placeholder feature names and input path before running the pipeline.

In [ ]:
# ---------- Files ----------
INPUT_FILE = Path(".csv") # change path to source
SEPARATOR = ","  # Examples: ",", ";", or "\t"
OUTPUT_DIR = Path("outputs")
CLUSTERED_DATA_FILE = OUTPUT_DIR / "data_umap_hdbscan.csv"
SETTINGS_FILE = OUTPUT_DIR / "umap_hdbscan_settings.json"
FIGURE_FILE = OUTPUT_DIR / "umap_hdbscan_3d.tiff"

# ---------- Variables ----------
FEATURES = [
    "feature_1", # change to input variables of choice
    "feature_2",
    "feature_3",
]
ID_COLUMNS = []  # Optional de-identified row keys to retain
MISSING_DATA_POLICY = "error"  # "error" or "complete_case"

# ---------- Models ----------
RANDOM_STATE = 0
UMAP_PARAMS = {
    "n_neighbors": 20,
    "min_dist": 0.01,
    "n_components": 3,
    "metric": "euclidean",
    "random_state": RANDOM_STATE,
}
HDBSCAN_PARAMS = {
    "min_cluster_size": 8,
    "min_samples": 5,
    "metric": "euclidean",
    "cluster_selection_method": "eom",
    "prediction_data": True,
}

COORDINATE_COLUMNS = ("UMAP1", "UMAP2", "UMAP3")
CLUSTER_COLUMN = "cluster"
MEMBERSHIP_COLUMN = "cluster_probability"
NOISE_LABEL = -1

## 4. Load and validate data

In [ ]:
if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")
if not FEATURES or len(FEATURES) != len(set(FEATURES)):
    raise ValueError("FEATURES must contain unique column names.")
if MISSING_DATA_POLICY not in {"error", "complete_case"}:
    raise ValueError("MISSING_DATA_POLICY must be 'error' or 'complete_case'.")

data = pd.read_csv(INPUT_FILE, sep=SEPARATOR)
required = [*ID_COLUMNS, *FEATURES]
missing_columns = [column for column in required if column not in data.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

numeric_features = data[FEATURES].apply(pd.to_numeric, errors="coerce")
incomplete = numeric_features.isna().any(axis=1)
if incomplete.any() and MISSING_DATA_POLICY == "error":
    raise ValueError(
        f"{int(incomplete.sum())} row(s) contain missing or non-numeric feature values."
    )

keep = ~incomplete if MISSING_DATA_POLICY == "complete_case" else pd.Series(True, index=data.index)
analysis_data = data.loc[keep].copy()
X_df = numeric_features.loc[keep].copy()
zero_variance = X_df.columns[X_df.nunique() <= 1].tolist()
if zero_variance:
    raise ValueError(f"Zero-variance feature(s): {zero_variance}")
if len(X_df) < HDBSCAN_PARAMS["min_cluster_size"]:
    raise ValueError("Analysis sample is smaller than min_cluster_size.")

print(f"Rows read: {len(data):,}")
print(f"Rows analyzed: {len(analysis_data):,}")
print(f"Features: {len(FEATURES)}")

## 5. Standardize features and fit 3D UMAP

In [ ]:
X_scaled = StandardScaler().fit_transform(X_df)
embedding = UMAP(**UMAP_PARAMS).fit_transform(X_scaled)

if embedding.shape[1] != 3:
    raise RuntimeError(f"Expected three UMAP dimensions; received {embedding.shape[1]}.")

for index, column in enumerate(COORDINATE_COLUMNS):
    analysis_data[column] = embedding[:, index]

## 6. Fit HDBSCAN and summarize assignments

In [ ]:
cluster_model = HDBSCAN(**HDBSCAN_PARAMS)
analysis_data[CLUSTER_COLUMN] = cluster_model.fit_predict(embedding).astype(int)
analysis_data[MEMBERSHIP_COLUMN] = cluster_model.probabilities_

cluster_summary = (
    analysis_data[CLUSTER_COLUMN]
    .value_counts()
    .rename_axis(CLUSTER_COLUMN)
    .reset_index(name="n")
    .sort_values(CLUSTER_COLUMN)
)
cluster_summary["percent_full_sample"] = 100 * cluster_summary["n"] / len(analysis_data)
cluster_summary

## 7. Plot clusters

In [ ]:
DEFAULT_COLORS = [
    "#0072B2", "#D55E00", "#009E73", "#CC79A7",
    "#E69F00", "#56B4E9", "#F0E442",
]
CLUSTER_COLORS = {}  # Optional: {0: "#0072B2", 1: "#D55E00"}
CLUSTER_NAMES = {}   # Optional: {0: "Profile A", 1: "Profile B"}

labels = sorted(analysis_data[CLUSTER_COLUMN].unique())
plot_order = [label for label in labels if label != NOISE_LABEL]
if NOISE_LABEL in labels:
    plot_order.append(NOISE_LABEL)

fig = plt.figure(figsize=(10, 8), constrained_layout=False)
ax = fig.add_subplot(111, projection="3d")
total = len(analysis_data)
x_col, y_col, z_col = COORDINATE_COLUMNS

for position, label in enumerate(plot_order):
    subset = analysis_data.loc[analysis_data[CLUSTER_COLUMN] == label]
    n = len(subset)
    percent = 100 * n / total
    name = CLUSTER_NAMES.get(
        label, f"Noise ({NOISE_LABEL})" if label == NOISE_LABEL else f"Cluster {label}"
    )
    color = CLUSTER_COLORS.get(
        label, "#BDBDBD" if label == NOISE_LABEL else DEFAULT_COLORS[position % len(DEFAULT_COLORS)]
    )
    ax.scatter(
        subset[x_col], subset[y_col], subset[z_col],
        s=40, c=color, label=f"{name}: N={n} ({percent:.1f}%)",
        alpha=0.35 if label == NOISE_LABEL else 0.85, edgecolors="none",
    )

ax.set_title("HDBSCAN clustering on a 3D UMAP embedding", fontsize=14, pad=20)
ax.set_xlabel(x_col, labelpad=10)
ax.set_ylabel(y_col, labelpad=10)
ax.set_zlabel(z_col, labelpad=10)
ax.view_init(elev=30, azim=-60)
ax.legend(loc="upper right", frameon=True, fontsize=10)
fig.subplots_adjust(left=0.02, right=0.95, bottom=0.05, top=0.95)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(
    FIGURE_FILE, dpi=600, bbox_inches="tight", pad_inches=0.3,
    pil_kwargs={"compression": "tiff_lzw"},
)
print(f"Saved figure: {FIGURE_FILE.resolve()}")
plt.show()

## 8. Export clustered data and settings

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
analysis_data.to_csv(CLUSTERED_DATA_FILE, index=False)
settings = {
    "features": FEATURES,
    "missing_data_policy": MISSING_DATA_POLICY,
    "umap": UMAP_PARAMS,
    "hdbscan": HDBSCAN_PARAMS,
    "n_rows_read": len(data),
    "n_rows_analyzed": len(analysis_data),
}
SETTINGS_FILE.write_text(json.dumps(settings, indent=2), encoding="utf-8")
print(f"Saved data: {CLUSTERED_DATA_FILE.resolve()}")
print(f"Saved settings: {SETTINGS_FILE.resolve()}")